# 02 — Preprocessing, feature reduction and leakage audit

The protocol fits **every** learned transformation on the training split only: median imputation,
one-hot vocabulary, standardisation, PCA loadings / mutual-information scores, and the final
min–max rescaling to `[0, π]`. Validation selects configuration and thresholds. The test split is
scored once.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
DATA = ROOT / "data"
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True


def show_image(path):
    """Render a saved PNG inside the notebook without extra display dependencies."""
    img = PILImage.open(path)
    plt.figure(figsize=(min(12, img.width / 100), min(8, img.height / 100)))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.close()


print("Project root:", ROOT)

## 1. What the preprocessor learned

In [ ]:
pre = json.loads((DATA / "processed" / "preprocessor.json").read_text(encoding="utf-8"))
print("Numeric columns:", len(pre["numeric_columns"]))
print("Categorical columns:", pre["categorical_columns"])
print("Dropped constants:", pre["constant_columns"])
print("Design-matrix features:", len(pre["feature_names"]))
print("Rare-level threshold:", pre.get("rare_threshold"))

The design matrix has 111 columns: 34 numeric features plus one-hot columns for
`protocol_type`, `service` and `flag`. Four constant columns are dropped. The one-hot vocabulary is
stored with the preprocessor so that the external file cannot introduce unseen columns.

## 2. Feature-reduction ablation (validation only)

In [ ]:
fs = json.loads((DATA / "metadata" / "feature_selection.json").read_text(encoding="utf-8"))
ablation = pd.DataFrame(fs["ablation"])
ablation.pivot(index="n_features", columns="method", values="f1").round(4)

In [ ]:
ax = ablation.pivot(index="n_features", columns="method", values="f1").plot(marker="o")
ax.set_xlabel("reduced dimensions")
ax.set_ylabel("validation F1 (logistic-regression probe)")
ax.set_title("PCA vs mutual information at 8/12/16 dimensions")
plt.tight_layout()
plt.show()

PCA-16 is the best measured configuration (validation F1 0.9295), but eight qubits is the
simulator cap used for the comparative study. The chosen configuration is therefore **PCA-8**
(validation F1 0.9161, 66.7% explained variance) for *all three models*, including the classical
baseline. The 1.3-point gap to PCA-16 is a disclosed constraint, not a test-set choice.

## 3. Reduced matrices

In [ ]:
arrays = np.load(DATA / "processed" / "arrays.npz")
for key in arrays.files:
    print(f"{key:14s} {str(arrays[key].shape):14s} {arrays[key].dtype}")

In [ ]:
X_train, X_val, X_test = arrays["X_train"], arrays["X_val"], arrays["X_test"]
for name, X in (("train", X_train), ("validation", X_val), ("test", X_test)):
    print(f"{name:10s} min={X.min():.4f} max={X.max():.4f} mean={X.mean():.4f} std={X.std():.4f}")

All three matrices lie in `[0, π]`. The quantum models and the classical CNN+LSTM receive the same
eight columns; no model gets extra features or a different scaling.

## 4. Eight-check leakage audit

In [ ]:
audit = json.loads((DATA / "metadata" / "leakage_audit.json").read_text(encoding="utf-8"))
rows = []
for check in audit["checks"]:
    rows.append({
        "check": check["check"],
        "passed": check["passed"],
        "detail": {k: v for k, v in check.items() if k not in ("check", "passed")},
    })
print("Overall audit passed:", audit["passed"])
pd.DataFrame(rows)[["check", "passed"]]

The audit verifies scaler statistics, the one-hot vocabulary, statelessness of the transform, the
reducer input/output dimensions, exact row overlap, split integrity, matrix ranges and PCA
monotonicity. All eight checks passed.

## 5. Visual check: PCA variance and class separation

In [ ]:
show_image(RESULTS / "figures" / "pca_explained_variance.png")
show_image(RESULTS / "figures" / "reduced_feature_space.png")

**Takeaways**

1. The same 8-column, `[0, π]` matrix is used by CNN+LSTM, QSVM and VQC.
2. Every fitted statistic comes from the training split; the external file is transformed with the
   saved vocabulary.
3. The audit is automated and reproducible, not a one-off manual check.
4. The eight-qubit cap is the reason PCA-8 was selected; this is stated as a limitation.